# DiD-BCF — B1_null (linearity_degree=1)

**Workstream B1 · canonical DiD (selection on unobservables)**

sharp null tau=0: size and coverage under H0

Fits DiD-BCF on the `B1_null` scenario at `linearity_degree=1` and reports
metrics for **both** the plain DiD-BCF posterior and the proposed **posterior
correction** (Algorithm 1 of the theory note), so the correction can be judged
directly. Panel: N=200, 4 pre + 4 post periods.

> **Colab:** upload just this notebook and *Run all* — the first cell installs the
> dependencies and the second clones the engine automatically.

In [1]:
# Colab: install the DiD-BCF dependencies (stochtree provides the BCF sampler).
%pip install -q stochtree scikit-learn joblib tqdm pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 40.6 MB/s eta 0:00:00


In [2]:
import os, sys

# --- Locate the DiD-BCF engine ------------------------------------------------
# So you can upload just THIS notebook to Colab and Run all. Resolution order:
#   1. `did_bcf_revision` already importable;
#   2. running inside a repo checkout (the parent folder holds the package);
#   3. otherwise clone https://github.com/hugogobato/DiD-BCF and use it.
REPO_URL = "https://github.com/hugogobato/DiD-BCF.git"
ENGINE_SUBDIR = os.path.join("DiD-BCF", "Simulation_Studies_Revision")

def _locate_root():
    try:
        import did_bcf_revision  # noqa: F401
        return os.path.dirname(os.path.dirname(did_bcf_revision.__file__))
    except Exception:
        pass
    parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if os.path.isdir(os.path.join(parent, "did_bcf_revision")):
        return parent
    if not os.path.isdir("DiD-BCF"):
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    return os.path.abspath(ENGINE_SUBDIR)

ROOT = _locate_root()
sys.path.insert(0, ROOT)
print("Using DiD-BCF engine at:", ROOT)

from did_bcf_revision.runner import run_named
from did_bcf_revision.metrics import (compute_metrics, plain_vs_corrected,
                                      surface_metrics)

Using DiD-BCF engine at: /content/DiD-BCF/Simulation_Studies_Revision


In [3]:
REPS = 100      # replications (lower for a quick smoke test)
JOBS = 1        # parallel reps (keep 1 on a single-core/GPU Colab)

bcf_params = dict(num_gfr=50, num_mcmc=500, keep_every=5, num_chains=3)

summaries = run_named(
    "B1_null",
    linearity_degree=1,
    reps=REPS,
    jobs=JOBS,
    bcf_params=bcf_params,
    prop_method="logit",   # pilot propensity for the posterior correction
    n_splits=2,            # cross-fitting folds for the correction
)
summaries.head()

[B1_null_lin_1] canonical DGP | N=(200,) | reps=100 | 100 fits | jobs=1


B1_null: 100%|██████████| 100/100 [4:25:30<00:00, 159.31s/fit]

[B1_null_lin_1] wrote 2000 rows -> /content/DiD-BCF/Simulation_Studies_Revision/Results/summaries_B1_null_lin_1.csv


,dgp,setting,linearity_degree,N,rep,estimand_type,estimand_id,g,t,k,...,p_bayes,surf_rmse,surf_mae,surf_n,surf_mape,surf_cover95,surf_len95,surf_cover90,surf_len90,true
0,canonical,B1_null,1,200,0,GATT,g=4_t=4,4.0,4.0,0.0,...,0.291333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
1,canonical,B1_null,1,200,0,GATT,g=4_t=5,4.0,5.0,1.0,...,0.296000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
2,canonical,B1_null,1,200,0,GATT,g=4_t=6,4.0,6.0,2.0,...,0.471333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
3,canonical,B1_null,1,200,0,GATT,g=4_t=7,4.0,7.0,3.0,...,0.474000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
4,canonical,B1_null,1,200,0,ES,k=0,NaN,NaN,0.0,...,0.291333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0


In [4]:
# Decomposed metrics: bias, MC SD/variance, RMSE, MAE, MAPE, coverage 90/95,
# interval length, calibration ratio (avg_post_sd/emp_sd), size/power and their
# Monte-Carlo SEs -- for plain AND corrected DiD-BCF.
metrics = compute_metrics(summaries)
plain_vs_corrected(metrics)

,dgp,setting,linearity_degree,N,estimand_type,estimand_id,role,bias_corrected,bias_plain,cover90_corrected,...,len95_corrected,len95_plain,mae_corrected,mae_plain,reject05_corrected,reject05_plain,rmse_corrected,rmse_plain,sd_ratio_corrected,sd_ratio_plain
0,canonical,B1_null,1,200,ATT,ATT,size,-0.030585,0.014130,0.46,...,0.705815,1.165849,0.359903,0.040621,0.44,0.0,0.449743,0.054279,0.397011,5.255201
1,canonical,B1_null,1,200,ES,k=0,size,-0.051294,-0.002136,0.58,...,0.882765,1.031953,0.371811,0.006022,0.34,0.0,0.465286,0.007378,0.485164,32.748221
2,canonical,B1_null,1,200,ES,k=1,size,-0.025455,0.012625,0.61,...,0.872304,1.214463,0.351751,0.046352,0.32,0.0,0.442233,0.061615,0.500414,4.767790
3,canonical,B1_null,1,200,ES,k=2,size,-0.025460,0.020956,0.56,...,0.901045,1.335004,0.382082,0.060685,0.38,0.0,0.475228,0.081787,0.481009,4.077540
4,canonical,B1_null,1,200,ES,k=3,size,-0.020131,0.025075,0.60,...,0.887130,1.439034,0.355763,0.068389,0.33,0.0,0.453437,0.091752,0.496162,3.994179
5,canonical,B1_null,1,200,GATT,g=4_t=4,size,-0.051294,-0.002136,0.58,...,0.882765,1.031953,0.371811,0.006022,0.34,0.0,0.465286,0.007378,0.485164,32.748221
6,canonical,B1_null,1,200,GATT,g=4_t=5,size,-0.025455,0.012625,0.61,...,0.872304,1.214463,0.351751,0.046352,0.32,0.0,0.442233,0.061615,0.500414,4.767790
7,canonical,B1_null,1,200,GATT,g=4_t=6,size,-0.025460,0.020956,0.56,...,0.901045,1.335004,0.382082,0.060685,0.38,0.0,0.475228,0.081787,0.481009,4.077540
8,canonical,B1_null,1,200,GATT,g=4_t=7,size,-0.020131,0.025075,0.60,...,0.887130,1.439034,0.355763,0.068389,0.33,0.0,0.453437,0.091752,0.496162,3.994179


## CATT-surface metrics (the paper's headline RMSE/MAE/MAPE)

Within-replication RMSE/MAE/MAPE over the *individual* treated observations
(mean +/- SD across runs) plus the *pointwise* CATT coverage -- the evidence
that DiD-BCF recovers the heterogeneous effect that GATT-only methods cannot.

In [5]:
surface_metrics(summaries)

/content/DiD-BCF/Simulation_Studies_Revision/did_bcf_revision/metrics.py:190: RuntimeWarning: Mean of empty slice
  rec[f"{c}_mean"] = float(np.nanmean(vals)) if vals.size else np.nan


,dgp,setting,linearity_degree,N,method,n_reps,avg_n_treated_obs,surf_rmse_mean,surf_rmse_sd,surf_mae_mean,...,surf_mape_mean,surf_mape_sd,surf_cover90_mean,surf_cover90_sd,surf_cover95_mean,surf_cover95_sd,surf_len90_mean,surf_len90_sd,surf_len95_mean,surf_len95_sd
0,canonical,B1_null,1,200,corrected,100,401.24,0.379203,0.260307,0.365351,...,NaN,NaN,0.5875,0.421,0.6575,0.415384,0.738747,0.088331,0.885811,0.109591
1,canonical,B1_null,1,200,plain,100,401.24,0.056086,0.040159,0.045390,...,NaN,NaN,1.0000,0.000,1.0000,0.000000,1.003515,0.059701,1.258568,0.071721
